In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Deploy your own System One model on Gemini platform

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/open-models/serving/gemini_platform_vllm_diffusiongemma_structured_outputs.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fopen-models%2Fserving%2Fgemini_platform_vllm_diffusiongemma_structured_outputs.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/open-models/serving/gemini_platform_vllm_diffusiongemma_structured_outputs.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/open-models/serving/gemini_platform_vllm_diffusiongemma_structured_outputs.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/open-models/serving/gemini_platform_vllm_diffusiongemma_structured_outputs.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/open-models/serving/gemini_platform_vllm_diffusiongemma_structured_outputs.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/open-models/serving/gemini_platform_vllm_diffusiongemma_structured_outputs.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/open-models/serving/gemini_platform_vllm_diffusiongemma_structured_outputs.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/open-models/serving/gemini_platform_vllm_diffusiongemma_structured_outputs.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>

| Author(s) |
| --- |
| [Ariel Jassan](https://github.com/arieljassan) |

## Overview

In this notebook, we will deploy, test, and benchmark the [**DiffusionGemma 26B-A4B**](https://huggingface.co/nvidia/diffusiongemma-26B-A4B-it-NVFP4) model on a Dedicated Endpoint on the Gemini platform, giving you a **System One** model capable of serving low-latency structured decision requests.

DiffusionGemma 26B-A4B is Google DeepMind's discrete block-diffusion language model adapted from Gemma 4 (`26B` total parameters, `4B` active per token). Unlike autoregressive models that decode one token at a time from left to right, DiffusionGemma uses **bidirectional attention** across a fixed-width token canvas (`64` tokens per block).

Using official upstream [**vLLM (`vllm/vllm-openai:v0.31.0`)**](https://hub.docker.com/r/vllm/vllm-openai), you can serve the NVFP4-quantized [`nvidia/diffusiongemma-26B-A4B-it-NVFP4`](https://huggingface.co/nvidia/diffusiongemma-26B-A4B-it-NVFP4) checkpoint on a single 24 GB **`NVIDIA_L4`** GPU (`g2-standard-8`) for **Single-Step Structured Diffusion Reads (System One)**:

- By seeding a partially masked JSON canvas (`diffusion_seed_canvas`, `diffusion_read_only: True`, `diffusion_max_steps: 1`, `diffusion_constrained: True`), vLLM evaluates all decision fields simultaneously in a **single forward pass** and returns calibrated marginal probabilities and Shannon entropy confidence scores:
  - **`choice`**: Categorical selection (`A`–`Z`) mapped back to your choice keys.
  - **`score`**: Ordinal rating (`0`–`9`) with expected value (`sum(i * p_i)`).
  - **`boolean`**: Binary/epistemic truth probability (`P(true) / (P(true) + P(false))`).

### What you'll cover

- Run the offline unit test suite ([`diffusiongemma_structured_outputs/tests/`](diffusiongemma_structured_outputs/tests/)) to verify schema compilation, request building, logprob parsing, and monitoring utilities.
- Mirror the official upstream vLLM container (`docker.io/vllm/vllm-openai:v0.31.0`) to Google Cloud Artifact Registry using Cloud Build and `crane`.
- Deploy (or reuse) `nvidia/diffusiongemma-26B-A4B-it-NVFP4` on a Dedicated Endpoint (`g2-standard-8` with `1x NVIDIA_L4`) via [`deploy_diffusiongemma.py`](diffusiongemma_structured_outputs/deploy_diffusiongemma.py).
- Compile a multi-question structured decision rubric (`DiffusionGemmaClient.build_canvas`) and execute a single-step structured diffusion read using `endpoint.raw_predict`.
- Run a **multi-minute concurrency sweep (`c = 1, 2, 4, 8, 16`)** and fetch **GPU and CPU utilization** from Google Cloud Monitoring (`google.cloud.monitoring_v3`) to see where hardware resources saturate.
- Evaluate accuracy across **4 standard benchmarks (`banking77`, `ag_news`, `boolq`, `sst5`)** and log results to **Google Cloud Experiments**.
- Clean up the deployed endpoint and model resources.

## Get started

### Install required packages

In [ ]:
%pip install --upgrade --quiet google-cloud-aiplatform google-cloud-monitoring transformers aiohttp datasets pandas python-dotenv

### Authenticate your notebook environment (Colab only)

If you're running this notebook on Google Colab, run the following cell to authenticate your environment.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Set Google Cloud project information and initialize the SDK

To get started using the Gemini platform, you must have an existing Google Cloud project and [enable the API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment). The cell below also adds the companion [`diffusiongemma_structured_outputs/`](diffusiongemma_structured_outputs/README.md) module directory to `sys.path`.

In [ ]:
from datetime import datetime, timezone
import importlib
import json
from pathlib import Path
import sys
import time

from google.cloud import aiplatform
import pandas as pd

# Add the companion module directory to sys.path when running from open-models/serving/.
MODULE_DIR = Path("diffusiongemma_structured_outputs")
if MODULE_DIR.is_dir() and str(MODULE_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR.resolve()))

import benchmark_diffusiongemma
import benchmark_load
import benchmark_quality
import config
import deploy_diffusiongemma
import diffusiongemma_client
import monitoring
import run_example

# fmt: off
PROJECT_ID = "[your-project-id]"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
LOCATION = "us-central1"  # @param {type: "string"}
ENDPOINT_NAME = "diffusiongemma-vllm"  # @param {type: "string"}
# fmt: on
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = config.resolve_default_project()

HF_MODEL_ID = config.HF_MODEL_ID
SERVED_MODEL_NAME = config.SERVED_MODEL_NAME

aiplatform.init(project=PROJECT_ID, location=LOCATION)
print(f"Initialized Gemini platform for project={PROJECT_ID} in {LOCATION}")

### Run the offline unit test suite

Verify schema compilation, vLLM request construction, logprob parsing, and monitoring utilities using the offline `unittest` suite before provisioning cloud resources:

In [ ]:
import unittest

# Reload companion modules and run all unit tests in tests/.
for mod in (
    config,
    diffusiongemma_client,
    deploy_diffusiongemma,
    run_example,
    benchmark_load,
    benchmark_quality,
    monitoring,
    benchmark_diffusiongemma,
):
    importlib.reload(mod)
for mod_name in list(sys.modules):
    if mod_name.startswith("test_"):
        sys.modules.pop(mod_name, None)

tests_dir = (
    str(MODULE_DIR / "tests")
    if (MODULE_DIR / "tests").is_dir()
    else "tests"
)
loader = unittest.TestLoader()
suite = loader.discover(start_dir=tests_dir, pattern="test_*.py")
runner = unittest.TextTestRunner(stream=sys.stdout, verbosity=2, buffer=True)
test_result = runner.run(suite)
assert test_result.wasSuccessful(), "Unit tests failed!"

## Mirror the Official vLLM Image to Artifact Registry

Online Prediction on the Gemini platform requires serving containers to reside in Google Cloud Artifact Registry (`*.pkg.dev`). Run the following cell once per project to create the `containers` repository and copy [`vllm/vllm-openai:v0.31.0`](https://hub.docker.com/r/vllm/vllm-openai) directly from Docker Hub to Artifact Registry inside Google Cloud using Cloud Build and [`crane`](https://github.com/google/go-containerregistry/blob/main/cmd/crane/README.md) (~20 seconds, no local Docker daemon required):

In [ ]:
!gcloud artifacts repositories create containers \
    --repository-format=docker \
    --location=us \
    --project={PROJECT_ID} || true

mirror_cfg = Path("mirror_vllm.yaml")
mirror_cfg.write_text(
    "steps:\n"
    "  - name: 'gcr.io/go-containerregistry/crane:latest'\n"
    "    args:\n"
    "      - 'copy'\n"
    "      - '--platform=linux/amd64'\n"
    "      - 'docker.io/vllm/vllm-openai:v0.31.0'\n"
    f"      - 'us-docker.pkg.dev/{PROJECT_ID}/containers/vllm-openai:v0.31.0'\n"
)
!gcloud builds submit --no-source --config=mirror_vllm.yaml --project={PROJECT_ID}
mirror_cfg.unlink(missing_ok=True)

## Deploy (or reuse) DiffusionGemma 26B-A4B on a Dedicated Endpoint

Next, register the official vLLM serving container in Model Registry and deploy it to a Dedicated Endpoint on the Gemini platform using a `g2-standard-8` (`1x NVIDIA_L4`, 24 GB VRAM) instance.

We call [`deploy_diffusiongemma.deploy_model_to_endpoint()`](diffusiongemma_structured_outputs/deploy_diffusiongemma.py), which configures vLLM with:
- `--diffusion-config={"canvas_length": 64}` for 64-token block diffusion.
- `--attention-backend=TRITON_ATTN` and `--limit-mm-per-prompt={"image": 0, "video": 0}` so the NVFP4-quantized 26B-A4B model fits comfortably in 24 GB VRAM.
- `--enable-prefix-caching`, `--max-logprobs=32`, and `--max-num-seqs=16` for concurrent batching of single-step structured diffusion reads on the GPU.

> **Tip**: [`deploy_model_to_endpoint()`](diffusiongemma_structured_outputs/deploy_diffusiongemma.py) is idempotent. If you already deployed the endpoint from the CLI (`python3 diffusiongemma_structured_outputs/deploy_diffusiongemma.py --project $PROJECT_ID`), running the cell below will immediately detect and reuse the active endpoint.

In [ ]:
endpoint = deploy_diffusiongemma.deploy_model_to_endpoint(
    project=PROJECT_ID,
    name=ENDPOINT_NAME,
    hf_model_id=HF_MODEL_ID,
    served_model_name=SERVED_MODEL_NAME,
)
raw_predict_url = benchmark_load.build_dedicated_raw_predict_url(endpoint)

print(f"Endpoint: {endpoint.resource_name}")
print(f"URL:      {raw_predict_url}")
print(
    "Cloud Console Monitoring URL:\n"
    f"https://console.cloud.google.com/vertex-ai/locations/{LOCATION}/endpoints/"
    f"{endpoint.name}?project={PROJECT_ID}&tab=monitoring"
)

## Single-Step Structured Diffusion Reads (`DiffusionGemmaClient`)

In vLLM `v0.31.0+`, DiffusionGemma supports **single-step structured diffusion** via `vllm_xargs` on `/v1/chat/completions` ([vllm-project/vllm#57250](https://github.com/vllm-project/vllm/pull/57250), [#58216](https://github.com/vllm-project/vllm/pull/58216)).

Instead of generating JSON tokens one by one, [`DiffusionGemmaClient`](diffusiongemma_structured_outputs/diffusiongemma_client.py) compiles a structured schema into a **seeded token canvas** (`diffusion_seed_canvas`) where structural JSON tokens (`{"q0": "`, `", "q1": `, etc.) are already known and only the answer slots contain `<|Reserved_2|>` (`token_id=69`) mask tokens:

- `diffusion_max_steps: 1`: Runs a single bidirectional forward pass across the canvas.
- `diffusion_read_only: True`: Reads the marginal log-probabilities at every masked slot without mutating the canvas.
- `diffusion_constrained: True`: Restricts vocabulary logits to `logprob_token_ids`.
- `diffusion_canvas_length`: Narrowed to the smallest multiple of 16 that fits the JSON template (e.g., `48` tokens for 4 questions).

Use `endpoint.raw_predict(use_dedicated_endpoint=True)` to evaluate an enterprise support ticket across four structured questions (`routing_team`, `severity_level`, `churn_or_expansion_risk`, and `requires_immediate_page`) simultaneously in a single forward pass:

In [ ]:
client = diffusiongemma_client.DiffusionGemmaClient(HF_MODEL_ID)
compiled_canvas = client.build_canvas(run_example.EXAMPLE_QUESTIONS)
request_payload = client.build_request(
    state=run_example.EXAMPLE_STATE,
    compiled=compiled_canvas,
    model=SERVED_MODEL_NAME,
)

t0 = time.perf_counter()
response = endpoint.raw_predict(
    body=json.dumps(request_payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    use_dedicated_endpoint=True,
)
elapsed_ms = (time.perf_counter() - t0) * 1000.0
response.raise_for_status()

parsed = client.parse_response(response.json(), compiled_canvas)
print(f"HTTP {response.status_code} | Client RTT: {elapsed_ms:.1f} ms")
print(json.dumps(parsed, indent=2))

## Multi-Minute Concurrency Sweep & GPU/CPU Saturation Analysis

To see where the `g2-standard-8` (`1x NVIDIA_L4` 24 GB) hardware saturates—and to ensure Google Cloud Monitoring's 60-second sampling captures hardware telemetry at every concurrency stage—we run `REQUESTS_PER_LEVEL = 600` requests at each concurrency level (`1, 2, 4, 8, 16`), wait `65s` for Cloud Monitoring metric ingestion, and join the hardware utilization time series onto each concurrency row.

In [ ]:
importlib.reload(benchmark_load)
importlib.reload(monitoring)

# Configure concurrency levels and requests per level.
# Set FORCE_RERUN_SWEEP = True to run a fresh load test, or False to reuse an
# already-completed sweep in memory when re-querying Cloud Monitoring metrics.
CONCURRENCIES = [1, 2, 4, 8, 16]
REQUESTS_PER_LEVEL = 600
METRICS_WAIT_SECONDS = 65
FORCE_RERUN_SWEEP = False

if FORCE_RERUN_SWEEP or "sweep_rows" not in globals() or not sweep_rows:
    sweep_start_utc = datetime.now(timezone.utc)
    token = benchmark_load.fetch_gcloud_bearer_token()

    sweep_metrics, sweep_rows = benchmark_load.run_concurrency_sweep(
        client=client,
        url=raw_predict_url,
        token=token,
        concurrencies=CONCURRENCIES,
        requests_per_level=REQUESTS_PER_LEVEL,
        model=SERVED_MODEL_NAME,
    )
    sweep_end_utc = datetime.now(timezone.utc)

    if METRICS_WAIT_SECONDS > 0:
        print(
            f"Waiting {METRICS_WAIT_SECONDS}s for Cloud Monitoring 60s hardware"
            " metrics ingestion..."
        )
        time.sleep(METRICS_WAIT_SECONDS)
else:
    sweep_start_utc = datetime.fromisoformat(sweep_rows[0]["start_utc"])
    sweep_end_utc = datetime.fromisoformat(sweep_rows[-1]["end_utc"])
    print(
        f"Reusing completed concurrency sweep ({sweep_start_utc.isoformat()}"
        f" to {sweep_end_utc.isoformat()})..."
    )

util_summary, util_timeseries = monitoring.fetch_endpoint_utilization_metrics(
    project=PROJECT_ID,
    endpoint_id=endpoint.name,
    start_time=sweep_start_utc,
    end_time=sweep_end_utc,
)
monitoring.attach_utilization_to_sweep(sweep_rows, util_timeseries)

print("\n=== Concurrency Sweep & Hardware Saturation Table ===")
sweep_df = pd.DataFrame(sweep_rows)
display_cols = [
    c
    for c in [
        "concurrency",
        "succeeded",
        "requests",
        "duration_s",
        "rps",
        "prompt_tokens_mean",
        "output_tokens_mean",
        "total_tokens",
        "tokens_per_s",
        "p50_ms",
        "p90_ms",
        "p95_ms",
        "p99_ms",
        "gpu_duty_cycle_pct",
        "gpu_mem_bw_pct",
        "gpu_tensor_active_pct",
        "gpu_power_w",
        "cpu_utilization_pct",
        "gpu_vram_gib",
        "ram_used_gib",
        "errors",
    ]
    if c in sweep_df.columns
]
display(sweep_df[display_cols])

if util_timeseries:
    print("\n=== Minute-by-Minute Cloud Monitoring Hardware Time Series ===")
    display(pd.DataFrame(util_timeseries))
else:
    print(
        "\nNo 60s Cloud Monitoring samples found yet for this window."
        " Try increasing REQUESTS_PER_LEVEL or re-querying in 60s."
    )

## Dataset Quality Benchmarks (`banking77`, `ag_news`, `boolq`, `sst5`) & Cloud Experiments

Evaluate structured diffusion quality across all four datasets:
- **`banking77`**: 26-intent banking customer query classification (`choice`, `A`–`Z`).
- **`ag_news`**: 4-way news topic classification (`choice`, `A`–`D`).
- **`boolq`**: Binary reading comprehension (`boolean`, `true`/`false`).
- **`sst5`**: 5-level fine-grained sentiment rating (`score`, `0`–`4` expected value).

Set `NUM_SAMPLES = 0` to evaluate **all 16,160 examples** across all four datasets (`3,080` Banking77 + `7,600` AG News + `3,270` BoolQ + `2,210` SST-5), or `100` / `1000` for a faster stratified sample.

In [ ]:
importlib.reload(benchmark_quality)
importlib.reload(monitoring)

# Set NUM_SAMPLES = 0 to run all examples across all 4 datasets, or e.g. 100 / 1000.
# Set FORCE_RERUN_EVAL = True to run a fresh evaluation, or False to reuse an
# already-completed quality_metrics dict in memory.
NUM_SAMPLES = 100
EVAL_CONCURRENCY = 4
FORCE_RERUN_EVAL = False

if FORCE_RERUN_EVAL or "quality_metrics" not in globals() or not quality_metrics:
    eval_start_utc = datetime.now(timezone.utc)
    quality_metrics: dict[str, float] = {}

    for task_name, fn in [
        ("banking77", benchmark_quality.evaluate_banking77),
        ("ag_news", benchmark_quality.evaluate_ag_news),
        ("boolq", benchmark_quality.evaluate_boolq),
        ("sst5", benchmark_quality.evaluate_sst5),
    ]:
        label = "all" if NUM_SAMPLES <= 0 else str(NUM_SAMPLES)
        print(
            f"Running {task_name}"
            f" (num_samples={label}, concurrency={EVAL_CONCURRENCY})..."
        )
        token = benchmark_load.fetch_gcloud_bearer_token()
        m = fn(
            client=client,
            url=raw_predict_url,
            token=token,
            num_samples=NUM_SAMPLES,
            concurrency=EVAL_CONCURRENCY,
            model=SERVED_MODEL_NAME,
        )
        quality_metrics.update(m)

    eval_end_utc = datetime.now(timezone.utc)
else:
    print(
        f"Reusing completed dataset evaluation ({eval_start_utc.isoformat()}"
        f" to {eval_end_utc.isoformat()})..."
    )

# Fetch GPU and CPU utilization recorded during the dataset evaluations.
eval_util_summary, eval_util_rows = monitoring.fetch_endpoint_utilization_metrics(
    project=PROJECT_ID,
    endpoint_id=endpoint.name,
    start_time=eval_start_utc,
    end_time=eval_end_utc,
)

print("\n=== Dataset Quality Benchmark Results (by Dataset & Metric) ===")
quality_df = pd.DataFrame(
    benchmark_quality.build_quality_summary_rows(quality_metrics)
).set_index("dataset")
display(quality_df.fillna("—"))

if eval_util_rows:
    print("\n=== Hardware Utilization During Quality Evaluation ===")
    display(pd.DataFrame(eval_util_rows))

In [ ]:
# Log the latest benchmark metrics to Google Cloud Experiments and compare historical runs.
EXPERIMENT_NAME = "diffusiongemma-benchmarks"
run_name = f"notebook-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}"

combined_metrics = {}
if "quality_metrics" in globals():
    combined_metrics.update(quality_metrics)
if "sweep_metrics" in globals():
    combined_metrics.update(sweep_metrics)
if "util_summary" in globals():
    combined_metrics.update(util_summary)

monitoring.log_benchmark_run_to_experiments(
    project=PROJECT_ID,
    region=LOCATION,
    experiment_name=EXPERIMENT_NAME,
    run_name=run_name,
    params={
        "model": SERVED_MODEL_NAME,
        "hf_model_id": HF_MODEL_ID,
        "endpoint_id": ENDPOINT_NAME,
        "num_samples": NUM_SAMPLES if "NUM_SAMPLES" in globals() else 0,
        "eval_concurrency": (
            EVAL_CONCURRENCY if "EVAL_CONCURRENCY" in globals() else 4
        ),
        "sweep_concurrencies": (
            ",".join(
                str(c)
                for c in (
                    CONCURRENCIES
                    if "CONCURRENCIES" in globals()
                    else [1, 2, 4, 8, 16]
                )
            )
        ),
        "requests_per_level": (
            REQUESTS_PER_LEVEL if "REQUESTS_PER_LEVEL" in globals() else 600
        ),
    },
    metrics=combined_metrics,
)
print(f"Logged experiment run: {run_name}")
display(aiplatform.get_experiment_df(EXPERIMENT_NAME))

## Cleaning up

To avoid incurring ongoing GPU charges, undeploy and delete the endpoint and model using [`deploy_diffusiongemma.delete_endpoint_and_model()`](diffusiongemma_structured_outputs/deploy_diffusiongemma.py) (or from the CLI via `python3 diffusiongemma_structured_outputs/deploy_diffusiongemma.py --delete`):

In [ ]:
deploy_diffusiongemma.delete_endpoint_and_model(name=ENDPOINT_NAME)